In [1]:
import pandas as pd
import numpy as np
import yfinance as yf

# print("Pandas:", pd.__version__)
# print("NumPy:", np.__version__)

data = {
    "Date": [
        "2026-01-07",
        "2026-01-05",
        "2026-01-08",
        "2026-01-06"
    ],
    "SPY": [601.5, 598.0, 603.2, 600.1]
}

df = pd.DataFrame(data)

# print(df)
# print(df.dtypes)

df["Date"] = pd.to_datetime(df["Date"])
# print(df)
# print(df.dtypes)

df = df.set_index("Date")
# print(df)
# print(type(df.index))

# df = df.sort_values(by="Date", ascending=True)
df = df.sort_index(ascending=True)
# print(df)

monthly_prices_df = df.resample("ME").last() # ME = Month End
# print(monthly_prices_df)
# print(monthly_prices_df.shape)

def download_data(ticker, start_date, end_date, auto_adjust=True):
    data = yf.download(ticker, start=start_date, end=end_date, auto_adjust=auto_adjust)
    # data = data.dropna().squeeze()
    return data

data_spy = download_data("SPY", "2015-01-01", "2026-01-01")
# print(data_spy.head())
# print(data_spy.columns)
# print(data_spy.dtypes)
# print(type(data_spy.index))

data_spy_close = data_spy["Close", "SPY"]
# print(data_spy_close.head())
# print(data_spy_close.dtypes)

# print(data_spy_close.isna().sum())
# print(data_spy_close.index.duplicated().sum())
# print(data_spy_close.index.is_monotonic_increasing)
# print(data_spy_close.index[0])
# print(data_spy_close.index[-1])
# print(data_spy_close.shape)
# print(data_spy_close.tail())

spy_monthly = data_spy_close.resample("ME").last()
# print(spy_monthly.tail())
# print(spy_monthly.shape)

spy_monthly_returns = spy_monthly.pct_change()
# print(spy_monthly_returns.tail())
# print(spy_monthly_returns.isna().sum())

spy_monthly_returns_clean = spy_monthly_returns.dropna()
# print(spy_monthly_returns_clean.shape)
# print(spy_monthly_returns_clean.isna().sum())

positive_returns = spy_monthly_returns_clean[spy_monthly_returns_clean > 0]
print(f"Positive monthly returns: {positive_returns.count()}")
# positive_returns.shape
print(f"Mean of positive monthly returns: {positive_returns.mean():.2%}")

next_month_returns = spy_monthly_returns_clean.shift(-1)
print(next_month_returns.tail())
print(next_month_returns.isna().sum())
next_month_returns_clean = next_month_returns.dropna()
print(next_month_returns_clean.isna().sum())

future_returns_after_positive = next_month_returns[spy_monthly_returns_clean > 0]
future_returns_after_positive.tail()
future_returns_after_positive.isna().sum()
future_returns_after_positive_clean = future_returns_after_positive.dropna()
print(future_returns_after_positive_clean.shape[0])
print(f"Future returns mean after positive month : {future_returns_after_positive_clean.mean():.2%}")

future_returns_after_negative = next_month_returns[spy_monthly_returns_clean < 0]
future_returns_after_negative_clean = future_returns_after_negative.dropna()
print(future_returns_after_negative.shape[0])
print(future_returns_after_negative.isna().sum())
print(f"Future returns mean after negative month : {future_returns_after_negative_clean.mean():.2%}")

mean_positive = future_returns_after_positive_clean.mean()
mean_negative = future_returns_after_negative_clean.mean()
momentum_spread = mean_positive - mean_negative
print(f"Momentum spread : {momentum_spread:.2%}")

# future_returns_after_positive_clean.shape[0] + future_returns_after_negative_clean.shape[0] == next_month_returns.shape[0]
print(future_returns_after_positive_clean.shape[0] + future_returns_after_negative_clean.shape[0])
print(next_month_returns.dropna().shape[0])
total_residual_returns = future_returns_after_positive_clean.shape[0] + future_returns_after_negative_clean.shape[0] - next_month_returns.dropna().shape[0]
print(total_residual_returns)

alignement_check = pd.DataFrame({
    "return_t" : spy_monthly_returns_clean,
    "return_t_plus_1" : next_month_returns
})

print(alignement_check.head(6))

[*********************100%***********************]  1 of 1 completed

Positive monthly returns: 92
Mean of positive monthly returns: 3.36%
Date
2025-08-31    0.035620
2025-09-30    0.023837
2025-10-31    0.001950
2025-11-30    0.000797
2025-12-31         NaN
Freq: ME, Name: (Close, SPY), dtype: float64
1
0
91
Future returns mean after positive month : 0.74%
39
0
Future returns mean after negative month : 2.08%
Momentum spread : -1.33%
130
130
0
            return_t  return_t_plus_1
Date                                 
2015-02-28  0.056204        -0.015706
2015-03-31 -0.015706         0.009834
2015-04-30  0.009834         0.012856
2015-05-31  0.012856        -0.020312
2015-06-30 -0.020312         0.022589
2015-07-31  0.022589        -0.060950


## Preliminary Findings & Limitations

Nous avons réalisé une première exploration des données financières afin de revoir et de consolider certaines notions Python, notamment la manipulation des dates et des séries temporelles.

Cette analyse préliminaire nous a permis d'obtenir les résultats suivants :

- Future returns mean after positive month : 0.74%

- Future returns mean after negative month : 2.08%

**Momentum spread : -1.33 point de pourcent (pp)**

Ces résultats indiquent que, sur l'actif étudié (SPY) et la période considérée (2015–2025), nous observons un potentiel phénomène de mean reversion plutôt que de trend following / momentum à un horizon mensuel.

Cependant, cette observation reste purement exploratoire. Nous n'avons pas encore évalué sa significativité statistique, sa stabilité temporelle ni sa robustesse sur d'autres échantillons.

De plus, nous n'avons pas défini ni backtesté de stratégie exploitable. Les coûts de transaction (transaction costs), le turnover, le slippage, le rebalancing, les conditions d'exécution et les risques associés n'ont donc pas encore été étudiés.

**Nous ne pouvons ainsi pas conclure à l'existence d'un signal prédictif statistiquement robuste ou économiquement exploitable.**

Enfin, ce notebook, ainsi que les deux suivants, sont principalement consacrés à la révision et à la consolidation de certaines compétences Python nécessaires à la recherche quantitative.

Ils ne constituent pas encore le cœur de notre projet de recherche sur le trend following et le time-series momentum, que nous aborderons dans les prochaines étapes de la Phase 3.
